In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# Conexão (Usando ':memory:' para rodar localmente sem precisar de credenciais. 
client = QdrantClient(":memory:")

# Criando a coleção
client.create_collection(
    collection_name="padroes_projeto",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

print("Collection 'padroes_projeto' created.")

In [ ]:
from qdrant_client.models import PointStruct
from fastembed import TextEmbedding

# Carrega o modelo de IA leve e eficiente
print("Loading AI model...")
model = TextEmbedding('BAAI/bge-small-en-v1.5')

# Catálogo de Design Patterns
design_patterns = [
    ("Singleton", "Ensures a class has only one instance and provides a global point of access to it.", "Criacional"),
    ("Observer", "Defines a one-to-many dependency between objects so that when one object changes state, all its dependents are notified and updated automatically.", "Comportamental"),
    ("Strategy", "Defines a family of algorithms, encapsulates each one, and makes them interchangeable.", "Comportamental"),
    ("Factory Method", "Defines an interface for creating an object, but lets subclasses decide which class to instantiate.", "Criacional"),
    ("Decorator", "Attaches additional responsibilities to an object dynamically, providing a flexible alternative to subclassing.", "Estrutural")
]

# Gerando os vetores
print("Generating vectors...")
# Juntamos o nome e a descrição para o modelo capturar o contexto completo
descriptions = [f"{item[0]}: {item[1]}" for item in design_patterns]
embeddings = list(model.embed(descriptions))

points = []
for i, embedding in enumerate(embeddings):
    points.append(PointStruct(
        id=i,
        vector=embedding.tolist(),
        payload={
            "pattern_name": design_patterns[i][0],
            "description": design_patterns[i][1],
            "category": design_patterns[i][2],
        }
    ))

# Enviando para a coleção 'padroes_projeto'
client.upsert(collection_name="padroes_projeto", points=points)

print("Success! The design patterns are now in your vector database.")

In [ ]:
# Definindo o problema que queremos resolver
query_text = "I need a way to notify multiple components when a state changes"

# Transforma a pergunta do desenvolvedor em um vetor
query_vector = list(model.embed([query_text]))[0]

# Executa a busca na coleção 'padroes_projeto'
print(f"Searching for: '{query_text}'...\n")

search_result = client.query_points(
    collection_name="padroes_projeto",
    query=query_vector.tolist(),
    limit=2
).points

# Mostra os resultados
print("Results found:")
for result in search_result:
    print(f"- {result.payload['pattern_name']} (Score: {result.score:.4f})")
    print(f"  Categoria: {result.payload['category']}")
    print(f"  Descrição: {result.payload['description']}\n")